# CardioSafe AI - Notebook 07b: diagnostics for the label evidence and RxNorm

Notebook 07 gave plausible tiers, but I cannot judge them without seeing the actual label sentences. This notebook **prints raw evidence** for the cases I am unsure about:

1. Sentences behind suspicious flags (bradycardia for azithromycin / ciprofloxacin / citalopram / amlodipine, QT for digoxin / escitalopram / donepezil, ...).
2. A full scan of chlorpromazine's labels: is the QT statement genuinely absent, or in a field we do not scan, or missed by our rules?
3. Raw RxNorm responses for the names that failed (Zofran, Ecosprin, Azithral) and for Zithromax (which worked) so I can see why.

**Please paste the raw printed output of cells 2, 3 and 4 as text** (select the output and copy it), not a summary: the exact sentences are what I need. Needs notebook 07 to have run (it reads its cache). Takes about 2 minutes, around 10 API requests.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, re, importlib
WORK = '/content/drive/MyDrive/CardioSafe'; KB = WORK + '/kb'
sys.path.insert(0, WORK)
import cardiosafe_kb as K
importlib.reload(K)
fda_cache = K.JsonCache(KB + '/cache_openfda.json')
print('cached drugs:', len([k for k in fda_cache.d if k.startswith('fda|')]))

In [ ]:
CASES = [('azithromycin', 'bradycardia'), ('ciprofloxacin', 'bradycardia'), ('citalopram', 'bradycardia'), ('amlodipine', 'bradycardia'),
         ('ondansetron', 'bradycardia'), ('digoxin', 'qt'), ('escitalopram', 'qt'), ('donepezil', 'qt'), ('erythromycin', 'qt'),
         ('haloperidol', 'av_block'), ('hydroxychloroquine', 'av_block'), ('atenolol', 'av_block')]
for drug, topic in CASES:
    rec = fda_cache.get(f'fda|{drug}|10')
    if not rec:
        print(f'\n=== {drug} / {topic}: NOT IN CACHE'); continue
    s = K.summarize_evidence(rec, topic)
    print(f'\n=== {drug.upper()} / {topic}: tier {s["tier"]} (labels used {s["n_labels"]}, with a positive statement {s["n_positive"]})')
    seen = set()
    for l in [l for l in rec['labels'] if l['single_ingredient']][:5]:
        for e in l['evidence'][topic]:
            t = e['text'][:320]
            if t in seen: continue
            seen.add(t); print(f'   [{e["group"]}/{e["cls"]}] {t}')
            if len(seen) >= 4: break
        if len(seen) >= 4: break
    if not seen: print('   (no sentences extracted)')

In [ ]:
def raw_scan(drug, pattern=K.TOPICS['qt'], limit=10):
    params = {'search': f'openfda.generic_name:"{drug}"', 'limit': limit, 'sort': 'effective_time:desc'}
    js = K.http_get(K.FDA_LABEL, params)
    if js and js.get('_status') == 400:
        params.pop('sort'); js = K.http_get(K.FDA_LABEL, params)
    labels = (js or {}).get('results', [])
    print(f'{drug}: retrieved {len(labels)} labels')
    for i, lab in enumerate(labels):
        o = lab.get('openfda', {})
        hits = {}
        for field, v in lab.items():
            if isinstance(v, list) and v and isinstance(v[0], str):
                found = [s for s in K.split_sentences(' '.join(v)) if pattern.search(s)]
                if found: hits[field] = found
        scanned = [f for f in K.SECTION_GROUP if f in lab]
        print(f'  label {i}: {o.get("generic_name")} | effective {lab.get("effective_time")} | single={K.label_meta(lab)["single_ingredient"]}')
        print(f'      scanned fields present: {scanned}')
        print(f'      fields containing a QT/torsade sentence (ANY field): {list(hits)}')
        for field, found in list(hits.items())[:2]:
            print(f'      ({field}) {found[0][:300]}')
raw_scan('chlorpromazine')
print()
raw_scan('digoxin')

In [ ]:
def show(js, n=650):
    print('   ', json.dumps(js)[:n] if js is not None else None)
for term in ['Zofran', 'Zithromax', 'Ecosprin', 'Azithral']:
    print(f'\n##### {term}')
    c = K.http_get(f'{K.RXNAV}/approximateTerm.json', {'term': term, 'maxEntries': 5})
    print('approximateTerm:'); show(c)
    cand = ((c or {}).get('approximateGroup') or {}).get('candidate') or []
    if cand:
        best = max(cand, key=lambda x: float(x.get('score') or 0)); r = best.get('rxcui')
        print(f'best candidate rxcui={r}'); print('properties:'); show(K.http_get(f'{K.RXNAV}/rxcui/{r}/properties.json'))
        print('related IN:'); show(K.http_get(f'{K.RXNAV}/rxcui/{r}/related.json', {'tty': 'IN'}))
    print('alternative: rxcui.json?name=...'); show(K.http_get(f'{K.RXNAV}/rxcui.json', {'name': term, 'search': 2}))
    print('alternative: drugs.json?name=...'); show(K.http_get(f'{K.RXNAV}/drugs.json', {'name': term}))

## Send me
The raw text printed by cells 2, 3 and 4. After I see the real sentences I will fix the extractor (adding the drug-effect / patient-risk-factor distinction) and choose the RxNorm fallback route; no guessing.